# PravahX — DualSPHysics Google Colab GPU Runner

This notebook runs 3D SPH dam break simulations on a **Google Colab GPU (NVIDIA Tesla T4 / V100 / A100)**.

### Workflow:
1. Verify GPU acceleration (`nvidia-smi`).
2. Download and set up DualSPHysics Linux x64 CUDA binaries.
3. Upload your prepared `dualsphysics_case.tar.gz` from PravahX.
4. Execute the simulation and generate cryptographic SHA-256 file manifest.
5. Download `dualsphysics_results.tar.gz` for import into PravahX (`mode='import'`).

In [ ]:
# Step 1: Check GPU
!nvidia-smi

In [ ]:
# Step 2: Download & Extract DualSPHysics v5.2 Linux Binaries
import os
import urllib.request
import zipfile

os.makedirs("/content/dualsphysics", exist_ok=True)
url_bin = (
    "https://github.com/DualSPHysics/DualSPHysics/releases/download/"
    "v5.2.1/DualSPHysics_v5.2.1_Linux_x64.zip"
)
zip_path = "/content/dualsphysics_bin.zip"
try:
    urllib.request.urlretrieve(url_bin, zip_path)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall("/content/dualsphysics")
except Exception:
    print("Using local DualSPHysics installation")
os.environ["PATH"] += ":/content/dualsphysics/bin/linux"

In [ ]:
# Step 3: Upload Case Package
import tarfile

from google.colab import files

print("Upload your dualsphysics_case.tar.gz prepared by PravahX:")
uploaded = files.upload()
for fn in uploaded:
    if fn.endswith(".tar.gz") or fn.endswith(".tgz"):
        with tarfile.open(fn, "r:gz") as tar:
            tar.extractall(path="/content/run")
        print(f"Extracted {fn} to /content/run")

In [ ]:
# Step 4: Execute Simulation on GPU
%%bash
cd /content/run
chmod +x run_colab.sh
./run_colab.sh

In [ ]:
# Step 5: Download Verified Results Package
import os

from google.colab import files

results_path = "/content/run/dualsphysics_results.tar.gz"
if os.path.exists(results_path):
    print("Downloading results package for PravahX import...")
    files.download(results_path)
else:
    print("Error: dualsphysics_results.tar.gz not found.")